# 01 · Data Exploration

First look at the collected sources and annotated facts. Reproduces the EDA tables in `docs/data/phase2_report.md`.

Run the pipeline first (`pipelines/collect_sources.py`, `extract_text.py`, `coverage.py`).

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'data'
services = pd.read_csv(DATA / 'metadata' / 'services.csv')
sources = pd.read_csv(DATA / 'metadata' / 'sources.csv')
service_sources = pd.read_csv(DATA / 'metadata' / 'service_sources.csv')
log = pd.read_csv(DATA / 'metadata' / 'collection_log.csv')
documents = pd.read_csv(DATA / 'processed' / 'documents.csv')
facts = pd.read_csv(DATA / 'processed' / 'annotations' / 'service_facts.csv', keep_default_na=False)
coverage = pd.read_csv(DATA / 'processed' / 'field_coverage.csv')

## Services and sources
Which kinds of services are in scope, and how trustworthy and in which language are their sources?

In [ ]:
display(services.groupby(['category', 'government_level']).size().rename('services'))
display(pd.crosstab(sources['tier'], sources['language'], margins=True))

In [ ]:
# Which services have an official (T1) source that actually states requirements?
linked = service_sources.merge(sources[['source_id', 'tier']], on='source_id')
has_primary = linked[(linked['tier'] == 'T1') & (linked['relevance'] == 'primary')]['service_slug'].unique()
services.assign(has_t1_primary=services['service_slug'].isin(has_primary))[['service_slug', 'category', 'has_t1_primary']]

## Collection outcomes
Latest attempt per source.

In [ ]:
latest = log.sort_values('collected_at').groupby('source_id').tail(1)
latest['outcome'].value_counts()

## Extracted text: how much Khmer is there?
Pages declared as English still carry Khmer content (FAQs), which matters for extraction and search.

In [ ]:
docs = documents.merge(sources[['source_id', 'language', 'tier']], on='source_id')
display(docs.groupby('method').agg(n=('chars', 'size'), median_chars=('chars', 'median'), mean_khmer=('khmer_char_ratio', 'mean')).round(2))
docs.groupby('language')['khmer_char_ratio'].describe().round(2)

## Annotated facts and field coverage
What do official sources state, and what do they leave out?

In [ ]:
stated = facts[facts['status'] == 'stated']
evidence_lang = stated['evidence'].map(lambda e: 'km' if any('\u1780' <= c <= '\u17ff' for c in e) else 'en')
display(evidence_lang.value_counts().rename('facts by evidence language'))
display(pd.crosstab(facts['service_slug'], facts['field']))
coverage.dropna(subset=['completeness'])